Download http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz and unzip into './Data/IMDB/files/aclImdb'

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from modules import data_loaders, learning
from modules.models import ClassificationNet

In [10]:
# n_hidden = 128
n_hidden = 1
n_emb = 300
seq_len = 200
maxlen = 200
save_fq = 200

batch_size = 128
index_from = 3
vocab_size = 20000
learning_rate = 0.0005
num_epochs = 1000

# args = sys.argv[1:]
# config = args[0]

config = 'LCLCCL'

In [4]:
base_data_path = 'Data/IMDB/files/'
paths_train = [base_data_path + "aclImdb/train/pos", base_data_path + "aclImdb/train/neg"]
paths_test = [base_data_path + "aclImdb/test/pos", base_data_path + "aclImdb/test/neg"]
labels = [1, 0]

(X_train, y_train, mask_train), (X_val, y_val, mask_val), (X_test, y_test, mask_test), vocab = data_loaders.process_imdb(paths_train, 
                                                                                                            paths_test, 
                                                                                                            labels,
                                                                                                            num_words=vocab_size,
                                                                                                            maxlen=maxlen)


In [ ]:
train_loader = data_loaders.ReviewsLoader(X=X_train, y=y_train, batch_size=batch_size, mask=mask_train, shuffle=True)
valid_loader = data_loaders.ReviewsLoader(X_val, y_val, batch_size, mask=mask_val)
test_loader = data_loaders.ReviewsLoader(X_test, y_test, batch_size, mask=mask_test)


In [6]:
def accuracy(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pred_labels = (prediction >= 0.5).to(target.dtype)
    return (pred_labels == target).float().mean()

def loss_function(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x = x.view(-1)
    x = x.flatten().clamp(1e-6, 1 - 1e-6).float()
    y = y.view(-1).float()
    return nn.functional.binary_cross_entropy(x, y)

In [12]:
class_net = ClassificationNet(vocab_size=vocab_size+index_from, 
                              n_emb=n_emb, 
                              n_hidden=n_hidden, 
                              num_classes=1, 
                              config=config)

optimizer = optim.Adam(class_net.parameters(), lr=learning_rate)

lm_trainer = learning.LMTrainer(model=class_net,
                                optimizer=optimizer,
                                criterion=loss_function,
                                val_criterion=accuracy,
                                num_epochs=num_epochs,
                                train_loader=train_loader,
                                valid_loader=valid_loader,
                                test_loader=test_loader)

lm_trainer.train()


epoch 0001/1000 | batch 0000/167 | base_loss 1.0311 | total_loss -144.6255 | tokens 128 


KeyboardInterrupt: 

profiling

In [15]:
import torch.profiler as profiler


def test_with_loader(model, loader):
    model.train()
    # -------------------
    # профилируем одну эпоху
    # -------------------
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CPU],
        record_shapes=True,
        with_stack=True
    ) as prof:
        for step, (x, y, _) in enumerate(loader):
            with profiler.record_function("model_inference"):
                logits = model(x)
            if step >= 2:
                break
    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))


model = ClassificationNet(vocab_size=vocab_size+index_from, 
                              n_emb=n_emb, 
                              n_hidden=n_hidden, 
                              num_classes=1, 
                              config=config)

test_with_loader(model, train_loader)


---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
            model_inference        28.67%     187.962ms        99.89%     654.888ms     218.296ms             3  
                  aten::mul        14.18%      92.956ms        17.13%     112.294ms      15.500us          7245  
           aten::randn_like         0.01%      69.800us        14.09%      92.344ms      15.391ms             6  
              aten::normal_        14.07%      92.243ms        14.07%      92.243ms       7.687ms            12  
                  aten::add         8.03%      52.669ms        10.33%      67.713ms       9.342us          7248  
                aten::slice         6.29%      41.212ms         7.11%      46.617ms     